# CI com GitHub Actions — Bella Tavola 🍝
## Parte 3: Integração com modelo e debugging

# BLOCO 5

**Objetivo:** Integrar o modelo do Hugging Face Hub ao pipeline de CI — baixando o artefato de forma segura, usando secrets do GitHub, e escrevendo testes para o endpoint `/ml/predict`. Ao final deste bloco, o pipeline verifica não apenas a API, mas também o modelo em produção.

## Conceitos-chave do Bloco 5

### O modelo no contexto de CI

Na Semana 2, você publicou o modelo no Hugging Face Hub e criou a função `load_model(force_download)`. Agora esse modelo precisa ser acessível durante o pipeline de CI.

Há duas abordagens possíveis:

| Abordagem | Vantagem | Desvantagem |
|---|---|---|
| Commitar o `model.pkl` no repositório Git | Simples, sem dependência externa | Arquivo binário no Git cresce o repositório; versão pode desincronizar do Hub |
| Baixar do Hugging Face Hub no pipeline | Sempre usa a versão publicada; sem binário no Git | Depende de rede e de token válido no CI |

Neste caderno, vamos com a segunda abordagem — baixar do Hub no pipeline. É mais realista e reforça o papel do Hub como registry central.

### Secrets no GitHub Actions

Um secret é uma variável de ambiente protegida. O GitHub:

- Nunca exibe o valor nos logs — mascara automaticamente com `***`
- Armazena de forma criptografada nas configurações do repositório
- **Não disponibiliza secrets para workflows disparados por forks externos** — isso é uma proteção de segurança importante

Essa última característica tem uma consequência prática: se o seu repositório for público e alguém de fora abrir um pull request, o step que usa `HF_TOKEN` vai falhar. Por isso, testes que dependem de secrets devem ser marcados como `integracao` e configurados para rodar apenas em pushes diretos ao branch principal — nunca em pull requests de origem desconhecida.

**Como configurar um secret no repositório:**

1. Abra o repositório no GitHub
2. Vá em **Settings → Secrets and variables → Actions**
3. Clique em **New repository secret**
4. Nome: `HF_TOKEN`
5. Valor: seu token do Hugging Face com permissão de leitura

**Como usar no workflow:**

```yaml
- name: Rodar testes de integração
  run: pytest tests/ -m integracao -v --tb=short
  env:
    HF_TOKEN: ${{ secrets.HF_TOKEN }}
```

A variável `HF_TOKEN` fica disponível como variável de ambiente para o processo Python durante aquele step:

```python
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)
```

### Cache de dependências no pipeline

Baixar o modelo a cada execução do pipeline adiciona tempo. O GitHub Actions oferece cache para evitar downloads repetidos:

```yaml
- name: Cache do modelo Hugging Face
  uses: actions/cache@v4
  with:
    path: ~/.cache/huggingface
    key: hf-model-${{ hashFiles('requirements.txt') }}
```

**Uma observação importante sobre essa chave de cache:** ela é invalidada quando o `requirements.txt` muda — ou seja, quando as dependências Python mudam. Ela **não** é invalidada quando o modelo no Hub é atualizado. Isso significa que o pipeline pode usar um modelo desatualizado sem perceber.

Para invalidar o cache quando o modelo mudar, inclua a versão explicitamente na chave:

```yaml
key: hf-model-v2-${{ hashFiles('requirements.txt') }}
```

Mude `v2` para `v3` quando publicar uma nova versão do modelo. O trade-off: cache mais agressivo acelera o pipeline, mas exige disciplina para invalidar manualmente quando o artefato muda.

### O contrato do modelo e da API

Antes de escrever testes para o `/ml/predict`, é importante ter clareza sobre dois contratos que precisam estar alinhados:

**Contrato do modelo:** quais features ele espera, em qual ordem, e com qual tipo de dado.

**Contrato da API:** quais campos o endpoint `/ml/predict` recebe, em qual ordem monta o array de features, e o que retorna.

Se esses dois contratos divergirem — mesmo que silenciosamente — o modelo vai predizer com features embaralhadas, sem nenhum erro aparente. O pipeline vai ficar verde, e as predições vão estar erradas.

## Checklist antes de testar o modelo no CI

Antes de escrever qualquer teste de integração, confirme localmente cada item abaixo. Se qualquer um falhar, resolva antes de continuar.

```bash
# 1. O model_utils.py existe e é importável
python -c "from model_utils import load_model; print('OK')"

# 2. O modelo carrega sem erro (com token configurado localmente)
export HF_TOKEN=hf_seu_token_aqui   # Linux/macOS
python -c "
from model_utils import load_model
m = load_model('seu-usuario/seu-repositorio')
print('Modelo carregado:', type(m))
"

# 3. O endpoint de predição existe e responde
# (com a API rodando localmente em outro terminal)
curl -X POST http://localhost:8000/ml/predict \
  -H "Content-Type: application/json" \
  -d '{"seu_campo": valor}'
```

## Exercício 5.1 — Configurando o secret HF_TOKEN

**Nível:** Essencial

**Conceito:** Criar secret no GitHub, verificar que o pipeline consegue autenticar.

### Sua tarefa

1. Crie o secret `HF_TOKEN` no seu repositório GitHub conforme as instruções acima.

2. Adicione um step de verificação ao `ci.yml` que confirma que o token está disponível **sem expô-lo**:

```yaml
- name: Verificar autenticação com Hugging Face
  run: python -c "import os; assert os.environ.get('HF_TOKEN'), 'HF_TOKEN não configurado'"
  env:
    HF_TOKEN: ${{ secrets.HF_TOKEN }}
```

3. Faça push e confirme que o step passa.

In [1]:
# Resposta:
# O step passou quando o secret HF_TOKEN estava configurado corretamente no repositório.
# Nos logs, o valor do token não ficou visível, porque o GitHub mascara secrets automaticamente.
# Se o secret não existir, a variável fica vazia e o step falha.

# Trecho do workflow com uso do secret HF_TOKEN

- name: Verificar autenticação com Hugging Face
  run: python -c "import os; assert os.environ.get('HF_TOKEN'), 'HF_TOKEN não configurado'"
  env:
    HF_TOKEN: ${{ secrets.HF_TOKEN }}

💡 Gabarito

In [ ]:
# @title
# ci.yml com step de verificação de autenticação
# Este step pertence apenas ao job de integração, não ao smoke.
# Secrets não estão disponíveis em PRs vindos de forks externos —
# por isso testes de integração com HF_TOKEN devem rodar
# apenas em pushes diretos ao branch main.
#
# name: CI — Bella Tavola
#
# on:
#   push:
#     branches: [main]
#   pull_request:
#     branches: [main]
#
# jobs:
#   testes-smoke:
#     runs-on: ubuntu-latest
#     steps:
#       - uses: actions/checkout@v4
#       - uses: actions/setup-python@v5
#         with:
#           python-version: "3.11"
#       - run: pip install --upgrade pip && pip install -r requirements.txt
#       - run: pytest tests/ -m smoke -v --tb=short
#
#   testes-integracao:
#     runs-on: ubuntu-latest
#     needs: testes-smoke
#     if: github.event_name == 'push' && github.ref == 'refs/heads/main'
#     steps:
#       - uses: actions/checkout@v4
#       - uses: actions/setup-python@v5
#         with:
#           python-version: "3.11"
#       - run: pip install --upgrade pip && pip install -r requirements.txt
#
#       - name: Verificar autenticação com Hugging Face
#         run: python -c "import os; assert os.environ.get('HF_TOKEN'), 'HF_TOKEN não configurado'"
#         env:
#           HF_TOKEN: ${{ secrets.HF_TOKEN }}
#
#       - name: Rodar testes de integração
#         run: pytest tests/ -m integracao -v --tb=short
#         env:
#           HF_TOKEN: ${{ secrets.HF_TOKEN }}
#
# O que observar:
# - O valor do token não aparece nos logs — o GitHub substitui por ***
# - O mascaramento acontece mesmo com print() no código Python
# - Isso protege contra exposição acidental, mas não contra código
#   malicioso que encode ou transforme o valor antes de exibi-lo
# - Se o secret não existir, a variável será uma string vazia
#   e o assert vai falhar com: AssertionError: HF_TOKEN não configurado

## Exercício 5.2 — Baixando o modelo no pipeline

**Nível:** Essencial

**Conceito:** Usar `load_model` no contexto de CI, verificar que o artefato é carregável.

> **Antes de continuar:** confirme que você completou o checklist da seção anterior. Em especial, `load_model(REPO_ID)` deve funcionar localmente com o token configurado.

### Referência

```python
# model_utils.py — função da Semana 2
import os
import joblib
from huggingface_hub import hf_hub_download, login

def load_model(
    repo_id: str,
    filename: str = "model.pkl",
    force_download: bool = False
):
    token = os.environ.get("HF_TOKEN")
    if token:
        login(token=token)

    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        force_download=force_download
    )
    return joblib.load(local_path)
```

### Sua tarefa

Crie o arquivo `tests/test_modelo.py` com testes que:

1. Carregam o modelo usando `load_model`
2. Verificam que o objeto retornado não é `None`
3. Verificam que o modelo tem os métodos `predict` e `predict_proba`
4. Fazem uma predição com uma amostra válida do domínio e confirmam o formato do resultado

Para o item 4, use uma amostra com valores plausíveis para o seu domínio — não apenas zeros. Zeros podem ser tecnicamente aceitos pelo modelo, mas em alguns casos resultam em comportamento inesperado se o modelo não viu esse padrão no treino.

> **Nota:** os testes deste bloco assumem `RandomForestClassifier`, que sempre possui `predict_proba`. Se você usou outro tipo de modelo, verifique se ele suporta esse método antes de incluir esse teste.

Marque todos os testes com `@pytest.mark.integracao`.

Antes de fazer push, teste localmente com o token configurado:

```bash
# Linux/macOS
export HF_TOKEN=hf_seu_token_aqui
pytest tests/test_modelo.py -v -m integracao

# Windows PowerShell
$env:HF_TOKEN="hf_seu_token_aqui"
pytest tests/test_modelo.py -v -m integracao
```

In [ ]:
# Trecho do arquivo: tests/test_modelo.py

import os
import pytest
import numpy as np

REPO_ID = os.getenv("HF_REPO_ID", "anaclaragr05/mlops-fraud-v1")


@pytest.fixture(scope="module")
def modelo():
    from model_utils import load_model
    return load_model(REPO_ID)


@pytest.fixture
def amostra_valida():
    return np.array([[150.0, 14, 5.0, 1, 0]])


@pytest.mark.integracao
def test_modelo_carregado_nao_e_none(modelo):
    assert modelo is not None


@pytest.mark.integracao
def test_modelo_tem_metodo_predict(modelo):
    assert hasattr(modelo, "predict")
    assert callable(modelo.predict)


@pytest.mark.integracao
def test_modelo_tem_metodo_predict_proba(modelo):
    assert hasattr(modelo, "predict_proba")
    assert callable(modelo.predict_proba)

💡 Gabarito

In [ ]:
# @title
# tests/test_modelo.py
import pytest
import numpy as np


REPO_ID = "seu-usuario/mlops-bella-tavola-v1"   # ajuste para seu repositório
N_FEATURES = 5                                   # ajuste para o número de features do seu modelo


@pytest.fixture(scope="module")
def modelo():
    """
    Carrega o modelo uma única vez para todos os testes deste módulo.
    scope="module" evita baixar o modelo a cada teste individual —
    importante para não sobrecarregar o pipeline com downloads repetidos.
    """
    from model_utils import load_model
    return load_model(REPO_ID)


@pytest.fixture
def amostra_valida():
    """
    Uma amostra sintética com valores plausíveis para o domínio.
    Adapte os valores para as features do seu modelo.
    Prefira valores dentro das faixas vistas no treinamento.
    """
    return np.array([[150.0, 14, 5.0, 1, 0]])   # ajuste para suas features


@pytest.mark.integracao
def test_modelo_carregado_nao_e_none(modelo):
    assert modelo is not None


@pytest.mark.integracao
def test_modelo_tem_metodo_predict(modelo):
    assert hasattr(modelo, "predict")
    assert callable(modelo.predict)


@pytest.mark.integracao
def test_modelo_tem_metodo_predict_proba(modelo):
    assert hasattr(modelo, "predict_proba")
    assert callable(modelo.predict_proba)


@pytest.mark.integracao
def test_predict_retorna_array_com_formato_correto(modelo, amostra_valida):
    resultado = modelo.predict(amostra_valida)
    assert resultado.shape == (1,)
    assert resultado[0] in [0, 1]


@pytest.mark.integracao
def test_predict_proba_retorna_probabilidades_validas(modelo, amostra_valida):
    probas = modelo.predict_proba(amostra_valida)
    assert probas.shape == (1, 2)               # duas classes
    assert abs(probas[0].sum() - 1.0) < 1e-6   # soma = 1
    assert all(0 <= p <= 1 for p in probas[0])  # cada valor entre 0 e 1


# Sobre scope="module":
# Por padrão, fixtures têm scope="function" — recriadas a cada teste.
# scope="module" significa que a fixture é criada uma vez e
# compartilhada por todos os testes do arquivo.
# Para o modelo, isso é importante: evita baixar o artefato
# repetidamente a cada teste individual.

## Exercício 5.3 — Testando o endpoint `/ml/predict`

**Nível:** Essencial

**Conceito:** Testar o endpoint de ML com `TestClient`, verificar estrutura e comportamento.

O endpoint `/ml/predict` recebe um payload JSON com as features, monta internamente um array numpy na ordem correta e chama o modelo. Esse processo de montagem é onde erros silenciosos acontecem com mais frequência.

Antes de escrever os testes, abra o arquivo `routers/predict.py` e localize o trecho onde o array de features é montado — algo como:

```python
features = np.array([[
    input.campo_1,
    input.campo_2,
    input.campo_3,
    ...
]])
```

Anote essa ordem. Os campos do payload nos testes precisam corresponder exatamente a ela.

### Contexto do Bella Tavola

O modelo do Bella Tavola foi treinado para prever o risco de um pedido ser problemático. As features refletem características do pedido e do histórico do cliente. Um exemplo de payload válido para esse contexto:

```python
PAYLOAD_VALIDO = {
    "valor_pedido": 120.0,          # valor total do pedido em reais
    "hora_pedido": 20,              # hora do dia (0-23)
    "num_itens": 3,                 # quantidade de pratos no pedido
    "historico_cancelamentos": 0,   # cancelamentos anteriores do cliente
    "distancia_entrega": 2.5        # km até o endereço de entrega
}
```

> **Adapte os campos para as features reais do seu modelo.** O que importa é que o payload corresponda exatamente ao `PredictInput` definido em `routers/predict.py` e à ordem de montagem do array de features.

### Sua tarefa

Adicione ao `tests/test_modelo.py` testes para o endpoint `/ml/predict` via `TestClient`:

1. Um payload válido retorna status 200
2. A resposta contém os campos `prediction`, `probability`, `label` e `model_version`
3. `prediction` é 0 ou 1
4. `probability` é um float entre 0 e 1
5. `label` é uma string não vazia
6. Um payload com campo obrigatório ausente retorna 422
7. Campos com valores fora do intervalo válido retornam 422

Marque todos com `@pytest.mark.integracao`.

In [ ]:
# Arquivo: aula_01_e02_fastapi/routers/ml.py

import os
import numpy as np

from fastapi import APIRouter, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

from model_utils import load_model

router = APIRouter()

MODEL_REPO_ID = os.getenv("HF_REPO_ID", "anaclaragr05/mlops-fraud-v1")
MODEL_FILENAME = os.getenv("HF_MODEL_FILENAME", "model.pkl")

_model = None


def get_model():
    global _model

    if _model is None:
        _model = load_model(
            repo_id=MODEL_REPO_ID,
            filename=MODEL_FILENAME
        )
    return _model


class PredictInput(BaseModel):
    valor_transacao: float = Field(gt=0)
    hora_transacao: int = Field(ge=0, le=23)
    distancia_ultima_compra: float = Field(ge=0)
    tentativas_senha: int = Field(ge=1)
    pais_diferente: int = Field(ge=0, le=1)


class PredictOutput(BaseModel):
    prediction: int
    probability: float
    label: str
    model_version: str


@router.post("/predict", response_model=PredictOutput)
async def predict(input_data: PredictInput):
    try:
        model = get_model()
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Modelo indisponível: {e}")

    features = np.array([[
        input_data.valor_transacao,
        input_data.hora_transacao,
        input_data.distancia_ultima_compra,
        input_data.tentativas_senha,
        input_data.pais_diferente
    ]], dtype=float)

    prediction = int(model.predict(features)[0])
    probability = float(model.predict_proba(features)[0][1])
    label = "fraude" if prediction == 1 else "legitimo"

    return PredictOutput(
        prediction=prediction,
        probability=round(probability, 4),
        label=label,
        model_version=MODEL_REPO_ID
    )


@router.get("/health")
async def health():
    try:
        model = get_model()
        test_input = np.zeros((1, 5))
        model.predict(test_input)

        return {
            "api": "ok",
            "model": "ok",
            "model_repo": MODEL_REPO_ID
        }

    except Exception as e:
        return JSONResponse(
            status_code=503,
            content={
                "api": "ok",
                "model": "degraded",
                "model_repo": str(e)
            }
        )

💡 Gabarito

In [ ]:
# @title
# Adicionar ao tests/test_modelo.py
# A fixture 'client' vem do conftest.py
import pytest


# Ajuste os campos para as features reais do seu modelo
PAYLOAD_VALIDO = {
    "valor_pedido": 120.0,
    "hora_pedido": 20,
    "num_itens": 3,
    "historico_cancelamentos": 0,
    "distancia_entrega": 2.5
}


@pytest.mark.integracao
def test_predict_retorna_200(client):
    response = client.post("/ml/predict", json=PAYLOAD_VALIDO)
    assert response.status_code == 200


@pytest.mark.integracao
def test_predict_retorna_campos_esperados(client):
    response = client.post("/ml/predict", json=PAYLOAD_VALIDO)
    assert response.status_code == 200
    dados = response.json()
    assert "prediction" in dados
    assert "probability" in dados
    assert "label" in dados
    assert "model_version" in dados


@pytest.mark.integracao
def test_predict_prediction_e_binario(client):
    response = client.post("/ml/predict", json=PAYLOAD_VALIDO)
    prediction = response.json()["prediction"]
    assert prediction in [0, 1]


@pytest.mark.integracao
def test_predict_probability_entre_zero_e_um(client):
    response = client.post("/ml/predict", json=PAYLOAD_VALIDO)
    probability = response.json()["probability"]
    assert isinstance(probability, float)
    assert 0.0 <= probability <= 1.0


@pytest.mark.integracao
def test_predict_label_e_string_nao_vazia(client):
    response = client.post("/ml/predict", json=PAYLOAD_VALIDO)
    label = response.json()["label"]
    assert isinstance(label, str)
    assert len(label) > 0


@pytest.mark.integracao
def test_predict_sem_campo_obrigatorio_retorna_422(client):
    payload_incompleto = {"valor_pedido": 120.0}   # faltam outros campos
    response = client.post("/ml/predict", json=payload_incompleto)
    assert response.status_code == 422


@pytest.mark.integracao
@pytest.mark.parametrize("campo,valor_invalido", [
    ("hora_pedido", 25),       # hora fora de 0-23
    ("hora_pedido", -1),       # hora negativa
    ("num_itens", 0),          # quantidade inválida
    ("valor_pedido", -50.0),   # valor negativo
])
def test_predict_campo_invalido_retorna_422(client, campo, valor_invalido):
    payload = {**PAYLOAD_VALIDO, campo: valor_invalido}
    response = client.post("/ml/predict", json=payload)
    assert response.status_code == 422

## Exercício 5.4 — Adicionando cache ao pipeline

**Nível:** Recomendado

**Conceito:** `actions/cache`, reduzir tempo de execução do pipeline.

### Sua tarefa

Adicione cache do Hugging Face ao job de integração do `ci.yml`. O cache deve armazenar o diretório `~/.cache/huggingface` e usar uma chave que reflita tanto o ambiente Python quanto a versão do modelo.

Faça duas execuções consecutivas do pipeline e compare os tempos:

**Para refletir:** Você publicou uma nova versão do modelo no Hub. O pipeline vai baixar automaticamente a versão nova, ou vai continuar usando o cache da versão anterior? O que você precisaria mudar no workflow para forçar o download?

In [ ]:
# Resposta:
# O cache reduz o tempo de execução do job de integração,
# porque evita baixar o modelo do Hub a cada execução.
# Isso deixa o pipeline mais rápido e mais eficiente.

- name: Cache do modelo Hugging Face
  uses: actions/cache@v4
  with:
    path: ~/.cache/huggingface
    key: hf-model-v1-${{ hashFiles('aula_01_e02_fastapi/requirements.txt') }}

💡 Gabarito

In [ ]:
# @title
# Trecho do job testes-integracao com cache
#
#       - name: Cache do modelo Hugging Face
#         uses: actions/cache@v4
#         with:
#           path: ~/.cache/huggingface
#           # A chave combina versão do modelo (manual) com hash das dependências.
#           # Mude 'v1' para 'v2' quando publicar uma nova versão do modelo
#           # — isso invalida o cache e força o download da versão nova.
#           key: hf-model-v1-${{ hashFiles('requirements.txt') }}
#
# Por que não confiar apenas no hashFiles('requirements.txt')?
# Porque o requirements.txt registra dependências Python, não o artefato.
# Um novo model.pkl no Hub não muda o requirements.txt,
# então o cache continuaria servindo o modelo antigo.
# A versão manual na chave é o mecanismo de controle.

## Exercício 5.5 — Desafio: testando comportamento do modelo 🎯

**Nível:** Desafio

**Conceito:** Testes de sanidade do modelo — verificar que o comportamento geral faz sentido.

Testes de estrutura verificam se a API responde corretamente. Testes de comportamento verificam se o modelo está fazendo o que se espera dele. São complementares.

**Um aviso importante antes de começar:** testes de comportamento são testes de **sanidade**, não provas de correção absoluta. Eles verificam que casos extremos e obviamente distintos geram respostas distintas. Se esses testes falharem após um retreinamento, isso pode significar que o modelo melhorou, piorou, ou que os dados mudaram — e a resposta correta é investigar, não apenas ajustar o threshold do `assert`.

### Sua tarefa

Construa dois payloads para o endpoint `/ml/predict`:

- Um caso que, pela lógica do seu domínio, deveria ter **baixa** probabilidade de ser positivo
- Um caso que, pela lógica do seu domínio, deveria ter **alta** probabilidade de ser positivo

Use os dados sintéticos da Semana 2 para entender quais combinações de features geram cada classe. Não invente valores arbitrários — construa a partir da lógica que foi usada no `gerar_dataset`.

Escreva um teste que verifica que a probabilidade do caso "obviamente positivo" é maior que a do caso "obviamente negativo".

In [ ]:
@pytest.mark.integracao
def test_modelo_distingue_casos_extremos(client):
    caso_tipico = {
        "valor_transacao": 55.0,
        "hora_transacao": 13,
        "distancia_ultima_compra": 3.0,
        "tentativas_senha": 1,
        "pais_diferente": 0
    }

    caso_suspeito = {
        "valor_transacao": 8900.0,
        "hora_transacao": 2,
        "distancia_ultima_compra": 450.0,
        "tentativas_senha": 6,
        "pais_diferente": 1
    }

    resp_tipico = client.post("/ml/predict", json=caso_tipico)
    resp_suspeito = client.post("/ml/predict", json=caso_suspeito)

    prob_tipico = resp_tipico.json()["probability"]
    prob_suspeito = resp_suspeito.json()["probability"]

    assert prob_suspeito > prob_tipico


@pytest.mark.integracao
def test_modelo_e_deterministico(client):
    payload = {
        "valor_transacao": 150.0,
        "hora_transacao": 14,
        "distancia_ultima_compra": 5.0,
        "tentativas_senha": 1,
        "pais_diferente": 0
    }

    resp_1 = client.post("/ml/predict", json=payload)
    resp_2 = client.post("/ml/predict", json=payload)

    assert resp_1.json()["prediction"] == resp_2.json()["prediction"]
    assert resp_1.json()["probability"] == resp_2.json()["probability"]

💡 Gabarito

In [ ]:
# @title
# Adicionar ao tests/test_modelo.py
# Adapte os valores para o domínio e features do seu modelo

@pytest.mark.integracao
def test_modelo_distingue_casos_extremos(client):
    """
    Teste de sanidade: verifica que o modelo atribui probabilidades
    diferentes para casos construídos para serem claramente distintos.

    Os valores abaixo foram escolhidos com base na lógica do gerar_dataset
    da Semana 2. Pedidos com valor alto, feitos tarde da noite,
    com histórico de cancelamentos, devem ter probabilidade maior
    de serem problemáticos do que pedidos típicos do horário de almoço.

    Adapte os payloads para as features do seu modelo.
    """
    caso_tipico = {
        "valor_pedido": 55.0,
        "hora_pedido": 13,
        "num_itens": 2,
        "historico_cancelamentos": 0,
        "distancia_entrega": 1.5
    }

    caso_suspeito = {
        "valor_pedido": 890.0,
        "hora_pedido": 2,
        "num_itens": 12,
        "historico_cancelamentos": 4,
        "distancia_entrega": 45.0
    }

    resp_tipico = client.post("/ml/predict", json=caso_tipico)
    resp_suspeito = client.post("/ml/predict", json=caso_suspeito)

    assert resp_tipico.status_code == 200
    assert resp_suspeito.status_code == 200

    prob_tipico = resp_tipico.json()["probability"]
    prob_suspeito = resp_suspeito.json()["probability"]

    assert prob_suspeito > prob_tipico, (
        f"Esperado: prob_suspeito ({prob_suspeito:.3f}) > "
        f"prob_tipico ({prob_tipico:.3f})\n"
        f"Se isso estiver falhando após retreinamento, investigue "
        f"antes de ajustar o teste."
    )


@pytest.mark.integracao
def test_modelo_e_deterministico(client):
    """O mesmo input deve sempre gerar o mesmo resultado."""
    resp_1 = client.post("/ml/predict", json=PAYLOAD_VALIDO)
    resp_2 = client.post("/ml/predict", json=PAYLOAD_VALIDO)
    assert resp_1.json()["prediction"] == resp_2.json()["prediction"]
    assert resp_1.json()["probability"] == resp_2.json()["probability"]

## Checklist do Bloco 5

Antes de seguir para o Bloco 6, confirme que você consegue:

- [ ] Criar e configurar um secret no repositório GitHub
- [ ] Usar `${{ secrets.HF_TOKEN }}` corretamente em um step do workflow
- [ ] Verificar que o token está disponível no pipeline sem expô-lo nos logs
- [ ] Explicar por que testes com secrets não devem rodar em PRs de forks
- [ ] Baixar e carregar o modelo do Hub dentro de um teste
- [ ] Testar estrutura e comportamento do endpoint `/ml/predict`
- [ ] Usar `scope="module"` em fixtures que carregam recursos pesados
- [ ] Configurar cache do Hugging Face e explicar quando ele é e não é invalidado
- [ ] Pipeline verde com testes de integração rodando no merge para main ✅

# BLOCO 6

**Objetivo:** Aprender a depurar pipelines que falham — lendo logs, identificando causas, corrigindo e prevenindo. Ao final deste bloco, você conseguirá diagnosticar qualquer falha comum em um pipeline de CI Python sem precisar adivinhar.

## Conceitos-chave do Bloco 6

### Por que pipelines falham de formas inesperadas

Um pipeline de CI roda em uma máquina limpa, sem histórico, sem estado, sem os arquivos que existem só na sua máquina. Isso expõe problemas que passam despercebidos localmente:

- Dependências instaladas globalmente na sua máquina mas não no `requirements.txt`
- Arquivos que existem localmente mas não foram commitados
- Variáveis de ambiente definidas no seu `.env` mas não configuradas como secrets
- Código que funciona na sua versão do Python mas não na versão do pipeline
- Testes que passam localmente porque o modelo já está em cache

A boa notícia: a maioria dessas falhas tem um padrão reconhecível nos logs.

### Como ler um log de pipeline com eficiência

Quando um pipeline falha, a tentação é ler tudo de cima para baixo. Uma heurística útil é ir direto ao **step que falhou** e ler o erro de baixo para cima dentro daquele step — na maioria dos casos, a última mensagem de erro é a mais informativa.

**Uma ressalva importante:** nem sempre o último erro é a causa raiz. Às vezes o erro visível é consequência de um problema que aconteceu antes:

```
✅ Instalar dependências
✅ Iniciar API em background    ← aqui a API falhou silenciosamente
❌ Verificar que a API responde ← aqui aparece o erro: "connection refused"
```

Nesse caso, a causa raiz está no step anterior. Por isso, quando o erro visível não fizer sentido imediato, suba um step e leia o log de lá.

**Estratégia geral:**

1. Identifique qual job falhou (❌ na interface)
2. Dentro do job, identifique qual step falhou
3. Expanda os logs daquele step e leia a última mensagem de erro
4. Se o erro não for claro, suba para o step anterior e repita

### Diferença entre `ERROR` e `FAILED` no pytest

Quando você lê logs do pytest, pode encontrar dois tipos de resultado negativo que significam coisas diferentes:

**`FAILED`** — o teste foi executado, mas um `assert` não passou:
```
FAILED tests/test_pratos.py::test_preco_invalido - AssertionError: assert 200 == 422
```

**`ERROR`** — o teste não chegou a executar porque houve um problema na coleta ou no setup — geralmente um erro de importação:
```
ERROR tests/test_modelo.py::test_predict_retorna_200
  ImportError while importing test module 'tests/test_modelo.py'
  ModuleNotFoundError: No module named 'model_utils'
```

A distinção importa para o diagnóstico: `FAILED` indica lógica incorreta no código ou no teste; `ERROR` indica problema estrutural de importação ou configuração.

### Mapa de falhas comuns

| Sintoma no log | Causa provável | Solução |
|---|---|---|
| `ModuleNotFoundError: No module named 'X'` | Biblioteca não está no `requirements.txt`, ou é código do projeto com problema de import path | Adicionar ao `requirements.txt` ou verificar estrutura de pastas |
| `ImportError: cannot import name 'X' from 'Y'` | Versão incompatível da biblioteca | Fixar versão no `requirements.txt` |
| `FileNotFoundError` | Arquivo não commitado ou caminho errado | Commitar o arquivo ou ajustar o caminho |
| `uvicorn: command not found` | uvicorn não está no `requirements.txt` | Adicionar uvicorn |
| `AssertionError` em um teste | Comportamento divergente do esperado | Corrigir o código ou o teste |
| `curl: (7) Failed to connect` | A API não subiu — ver step anterior | Verificar logs do step de inicialização |
| `HF_TOKEN não configurado` | Secret não criado ou não passado com `env:` | Criar secret e adicionar `env:` ao step |
| `401 Unauthorized` ao acessar Hub | Token inválido ou expirado | Regenerar token no Hugging Face |
| `X: command not found` (black, autoflake etc.) | Ferramenta não instalada no job | Adicionar ao `requirements.txt` ou instalar no step |

### `continue-on-error` e `if: failure()`

Em alguns casos, você quer que o pipeline continue mesmo depois de uma falha — para coletar mais informações:

```yaml
- name: Rodar testes
  run: pytest tests/ -v --tb=short
  continue-on-error: true   # o pipeline continua mesmo se este step falhar

- name: Salvar log de falhas
  if: failure()             # só roda se algum step anterior falhou
  run: pytest tests/ --tb=long > relatorio_falhas.txt
```

Use com moderação — um pipeline que "continua mesmo com falha" pode mascarar problemas reais.

## Exercício 6.1 — Lendo um log de falha

**Nível:** Essencial

**Conceito:** Interpretar logs, distinguir `ERROR` de `FAILED`, identificar causa raiz.

Analise o log de pipeline abaixo e responda às perguntas.

```
Run pytest tests/ -v --tb=short
collecting ... collected 8 items

tests/test_pratos.py::test_listar_pratos_retorna_200 PASSED
tests/test_pratos.py::test_listar_pratos_retorna_lista PASSED
tests/test_pratos.py::test_filtro_por_categoria PASSED
tests/test_pratos.py::test_buscar_prato_existente PASSED
tests/test_pratos.py::test_buscar_prato_inexistente_retorna_404 PASSED
tests/test_pedidos.py::test_criar_pedido_com_prato_existente PASSED
tests/test_pedidos.py::test_valor_total_calculado_corretamente PASSED
ERROR tests/test_modelo.py::test_predict_retorna_200

======== ERRORS ========
ERROR tests/test_modelo.py::test_predict_retorna_200
  ImportError while importing test module
  '/home/runner/work/bella-tavola/tests/test_modelo.py'
  File "tests/test_modelo.py", line 3, in <module>
    from model_utils import load_model
  ModuleNotFoundError: No module named 'model_utils'

======== short test summary info ========
ERROR tests/test_modelo.py::test_predict_retorna_200 - ModuleNotFoundError
1 error in 7 passed
```

In [ ]:
# Resposta:
# "1 error in 7 passed" significa que sete testes passaram,
# mas houve um erro estrutural antes da execução de um dos testes.
#
# Isso é diferente de FAILED:
# - FAILED: o teste executa e um assert falha
# - ERROR: o teste nem chega a rodar, normalmente por falha de importação,
#   fixture, configuração ou inicialização

💡 Gabarito

In [ ]:
# @title
# 1. 7 testes passaram. "1 error in 7 passed" significa que o pytest
#    conseguiu coletar e executar 7 testes com sucesso, mas encontrou
#    um ERROR ao tentar importar o módulo test_modelo.py.
#    O teste nem chegou a executar — o problema foi no import.

# 2. É um caso de ERROR, não FAILED.
#    FAILED = o teste executou, mas um assert não passou.
#    ERROR = o teste não chegou a executar porque houve problema
#    na importação ou configuração do módulo.
#    Para o pipeline, ambos resultam em código de saída != 0.
#    Para o diagnóstico, ERROR indica problema estrutural;
#    FAILED indica problema de lógica.

# 3. Hipóteses para o ModuleNotFoundError:
#    a) model_utils.py não foi commitado no repositório
#    b) model_utils.py existe mas está em uma subpasta,
#       e o pytest está rodando de uma pasta onde não o encontra
#    c) O arquivo existe mas tem nome diferente (ex: models_utils.py)
#    d) O projeto usa estrutura src/ e o PYTHONPATH não está configurado
#    e) O arquivo não foi adicionado ao git (git add esquecido)

# 4. Como verificar cada hipótese localmente:
#    a) ls -la model_utils.py  → arquivo existe?
#    b) git status  → arquivo está rastreado pelo git?
#    c) git log --oneline model_utils.py  → foi commitado?
#    d) python -c "from model_utils import load_model"  → import funciona?
#    e) find . -name "model_utils.py"  → onde está o arquivo?

# 5. Se fosse FAILED em vez de ERROR, indicaria que:
#    - o import funcionou (model_utils.py foi encontrado)
#    - o teste foi executado até o assert
#    - o assert falhou — problema de lógica no código ou no teste
#    O diagnóstico seria completamente diferente.

## Exercício 6.2 — Pipeline com problemas

**Nível:** Essencial

**Conceito:** Diagnosticar erros objetivos e riscos de engenharia em um workflow real.

O workflow abaixo tem problemas de dois tipos diferentes:

- **Erros objetivos:** causam falha imediata ou comportamento incorreto garantido
- **Riscos de engenharia:** não causam falha imediata, mas são más práticas que podem gerar problemas no futuro

Identifique **3 erros objetivos** e **2 riscos de engenharia** antes de ver o gabarito.

```yaml
name: CI Bella Tavola
on:
  push:
    branches: [main]

jobs:
  testes:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v2

      - uses: actions/setup-python@v2
        with:
          python-version: 3.9

      - name: Instalar dependências
        run: pip install fastapi uvicorn pytest

      - name: Verificar formatação
        run: black --check .

      - name: Rodar testes smoke
        run: pytest tests/ -m smoke -v
        env:
          HF_TOKEN: ${{ secret.HF_TOKEN }}

  publicar:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Publicar modelo
        run: python scripts/publicar.py
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
```

In [ ]:
# Resposta:
# Erros objetivos:
# 1. usar ${{ secret.HF_TOKEN }} em vez de ${{ secrets.HF_TOKEN }}
# 2. instalar dependências manualmente sem usar requirements.txt
# 3. rodar black sem instalar black
#
# Riscos de engenharia:
# 1. usar versões antigas de checkout e setup-python
# 2. publicar artefato sem depender do sucesso dos testes

💡 Gabarito

In [ ]:
# @title
# ERROS OBJETIVOS:

# ERRO 1: ${{ secret.HF_TOKEN }} — falta o 's' em 'secrets'
# O correto é ${{ secrets.HF_TOKEN }}
# Esse erro é silencioso: a variável será passada como string vazia.
# O pipeline só vai falhar quando o código tentar usar o token,
# não no step de configuração — o que dificulta o diagnóstico.

# ERRO 2: pip install fastapi uvicorn pytest — não usa requirements.txt
# Se uma nova biblioteca for adicionada ao projeto mas não a esse comando,
# o pipeline vai falhar com ModuleNotFoundError de forma não óbvia.
# Correto: pip install -r requirements.txt

# ERRO 3: black --check . — mas black não foi instalado
# O step de instalação só instala fastapi, uvicorn e pytest.
# black não está nem no requirements.txt nem no comando de instalação.
# Resultado: "black: command not found" e falha imediata do step.
# Solução: adicionar black ao requirements.txt

# ---

# RISCOS DE ENGENHARIA:

# RISCO 1: actions/checkout@v2 e actions/setup-python@v2 — versões antigas
# Não causam falha imediata, mas versões antigas podem ter comportamentos
# descontinuados, vulnerabilidades de segurança conhecidas ou
# incompatibilidades futuras com o ambiente do runner.
# Recomendado: usar @v4 e @v5 respectivamente.

# RISCO 2: job 'publicar' sem 'needs: testes'
# Os dois jobs rodam em paralelo. Isso significa que o modelo pode ser
# publicado mesmo que os testes estejam falhando — exatamente o oposto
# do que CI/CD deveria garantir.
# Correto: adicionar 'needs: testes' ao job 'publicar'.

# ---

# WORKFLOW CORRIGIDO:
#
# name: CI Bella Tavola
# on:
#   push:
#     branches: [main]
#
# jobs:
#   testes:
#     runs-on: ubuntu-latest
#     steps:
#       - uses: actions/checkout@v4             # atualizado
#       - uses: actions/setup-python@v5         # atualizado
#         with:
#           python-version: "3.11"
#       - name: Instalar dependências
#         run: |
#           pip install --upgrade pip
#           pip install -r requirements.txt     # corrige erro 2
#           # garanta que black está no requirements.txt (corrige erro 3)
#       - name: Verificar formatação
#         run: black --check .
#       - name: Rodar testes smoke
#         run: pytest tests/ -m smoke -v --tb=short
#         env:
#           HF_TOKEN: ${{ secrets.HF_TOKEN }}   # corrige erro 1
#
#   publicar:
#     runs-on: ubuntu-latest
#     needs: testes                             # corrige risco 2
#     steps:
#       - uses: actions/checkout@v4
#       - name: Publicar modelo
#         run: python scripts/publicar.py
#         env:
#           HF_TOKEN: ${{ secrets.HF_TOKEN }}

## Exercício 6.3 — Reproduzindo uma falha localmente

**Nível:** Essencial

**Conceito:** Simular o ambiente do CI localmente para depurar sem fazer push.

Uma das frustrações mais comuns com CI é o ciclo lento de depuração: editar → commitar → push → esperar o pipeline → ver o erro → repetir.

A alternativa é reproduzir o ambiente do CI localmente antes de fazer push — criando um ambiente virtual limpo que simula a máquina do runner.

### Sua tarefa

**Passo 1:** Crie um ambiente virtual limpo, sem herdar pacotes globais:

```bash
python -m venv env_ci_test
```

**Passo 2:** Ative o ambiente:

```bash
# Linux/macOS
source env_ci_test/bin/activate

# Windows PowerShell
env_ci_test\Scripts\activate
```

**Passo 3:** Instale apenas o que está no `requirements.txt`:

```bash
pip install --upgrade pip
pip install -r requirements.txt
```

**Passo 4:** Rode os testes exatamente como o pipeline faz:

```bash
pytest tests/ -v --tb=short
```

**Passo 5:** Se um teste falhar, você verá o mesmo erro que apareceria no pipeline — mas sem precisar fazer push.

**Passo 6:** Após depurar, desative e remova o ambiente temporário:

```bash
# Linux/macOS
deactivate
rm -rf env_ci_test

# Windows PowerShell
deactivate
Remove-Item -Recurse -Force env_ci_test
```

In [ ]:
# Resposta:
# Em um ambiente limpo, podem aparecer erros como:
# - ModuleNotFoundError para bibliotecas que estavam instaladas globalmente
# - incompatibilidade de versão de dependências
# - testes frágeis que dependem de estado absoluto
#
# Reproduzir localmente em ambiente limpo ajuda a aproximar o diagnóstico
# do comportamento real do CI.

💡 Gabarito

In [ ]:
# @title
# Erros comuns que aparecem no ambiente limpo e não localmente:

# 1. ModuleNotFoundError para ferramentas instaladas globalmente
#    Ex: black, autoflake, jupyter instalados fora do projeto
#    → Adicionar ao requirements.txt se forem usadas no pipeline

# 2. ImportError por versão incompatível
#    Ex: código usa sintaxe do pydantic v2, mas sem versão fixada
#    o pip instala v1
#    → Fixar versão: pydantic>=2.0,<3.0

# 3. FileNotFoundError para arquivos de configuração
#    Ex: .env carregado pelo pydantic-settings, mas o arquivo
#    .env não existe no ambiente limpo (e não deve ser commitado)
#    → Garantir que BaseSettings tem valores padrão para ambiente de teste

# 4. AssertionError em testes que dependem de dados pré-existentes
#    Ex: teste que verifica ID específico, mas no ambiente limpo
#    os dados ainda não foram criados por outro teste
#    → Reescrever o teste para não depender de estado absoluto

# A estratégia do ambiente virtual limpo é a forma mais confiável
# de reproduzir o CI localmente — e reduz drasticamente o ciclo
# de depuração via push.

## Exercício 6.4 — Adicionando diagnóstico ao pipeline

**Nível:** Recomendado

**Conceito:** Logs informativos, diagnóstico proativo no pipeline.

Logs genéricos dificultam o diagnóstico. Adicionar informações de contexto ao início do pipeline acelera a identificação de problemas.

### Referência

```yaml
- name: Diagnóstico do ambiente
  run: |
    echo "=== Python ==="
    python --version
    echo "=== Dependências instaladas ==="
    pip list
    echo "=== Arquivos do projeto ==="
    find . -name "*.py" | grep -v __pycache__ | sort
```

### Sua tarefa

Adicione um step de diagnóstico ao seu `ci.yml` que exibe, no início do pipeline:

1. Versão do Python
2. Lista de pacotes instalados
3. Confirmação de que os arquivos centrais do **seu** projeto existem (ajuste para os nomes reais)
4. Se `HF_TOKEN` está definido — sem exibir o valor

O step deve vir **depois** da instalação de dependências e **antes** dos testes. Ele não deve falhar o pipeline — é puramente informativo.

> **Ajuste os arquivos verificados** para os que fazem sentido no seu projeto específico — não copie cegamente os nomes do gabarito.

In [ ]:
- name: Diagnóstico do ambiente
  run: |
    echo "=== Python ==="
    python --version

    echo "=== Dependências instaladas ==="
    pip list

    echo "=== Arquivos do projeto ==="
    test -f aula_01_e02_fastapi/main.py          && echo "✅ main.py"          || echo "❌ main.py ausente"
    test -f aula_01_e02_fastapi/requirements.txt && echo "✅ requirements.txt" || echo "❌ requirements.txt ausente"
    test -d aula_01_e02_fastapi/tests            && echo "✅ tests/"           || echo "❌ tests/ ausente"
    test -f aula_01_e02_fastapi/model_utils.py   && echo "✅ model_utils.py"   || echo "❌ model_utils.py ausente"

    echo "=== Autenticação ==="
    if [ -n "$HF_TOKEN" ]; then
      echo "✅ HF_TOKEN definido"
    else
      echo "⚠️ HF_TOKEN não definido"
    fi
  env:
    HF_TOKEN: ${{ secrets.HF_TOKEN }}

💡 Gabarito

In [ ]:
# @title
# Trecho do ci.yml com step de diagnóstico
#
#       - name: Diagnóstico do ambiente
#         run: |
#           echo "=== Python ==="
#           python --version
#
#           echo "=== Dependências instaladas ==="
#           pip list
#
#           echo "=== Arquivos do projeto ==="
#           # Ajuste os nomes abaixo para os arquivos centrais do seu projeto
#           test -f main.py          && echo "✅ main.py"          || echo "❌ main.py ausente"
#           test -f requirements.txt && echo "✅ requirements.txt" || echo "❌ requirements.txt ausente"
#           test -d tests/           && echo "✅ tests/"           || echo "❌ tests/ ausente"
#           test -f model_utils.py   && echo "✅ model_utils.py"   || echo "❌ model_utils.py ausente"
#
#           echo "=== Autenticação ==="
#           if [ -n "$HF_TOKEN" ]; then
#             echo "✅ HF_TOKEN definido"
#           else
#             echo "⚠️  HF_TOKEN não definido (esperado em jobs de integração)"
#           fi
#         env:
#           HF_TOKEN: ${{ secrets.HF_TOKEN }}
#
# Este step usa || echo "..." em vez de || exit 1 intencionalmente.
# Ele reporta problemas sem interromper o pipeline —
# a falha real vai aparecer no step de testes, com mensagem mais precisa.
# Se você quiser que o diagnóstico falhe o pipeline, substitua
# '|| echo "❌ ..."' por '|| exit 1'.

## Exercício 6.5 — Desafio: pipeline de ponta a ponta 🏆

**Nível:** Desafio

**Conceito:** Integrar tudo em um único workflow completo e robusto.

Este é o exercício final do caderno. Você vai montar o `ci.yml` definitivo do Bella Tavola integrando todos os conceitos das três partes.

### Especificação do workflow

O workflow deve ter **três jobs** com a seguinte lógica:

**Job `qualidade`** — roda em todo push e pull request:
- Instala dependências do `requirements.txt` — que deve incluir `black` e `autoflake`
- Step de diagnóstico do ambiente
- Verifica formatação com `black --check`
- Verifica imports não utilizados com `autoflake --check`
- Roda testes smoke: `pytest tests/ -m smoke`

**Job `integracao`** — roda apenas em push para `main`, depois de `qualidade` passar:
- Configura cache do Hugging Face
- Step de diagnóstico com verificação do token
- Verifica autenticação com o Hub
- Roda testes de integração: `pytest tests/ -m integracao`

**Job `relatorio`** — roda apenas em push para `main`, depois de `integracao` passar:
- Exibe um resumo do pipeline com informações do commit
- Este job é um **gancho didático** — em um projeto real, aqui entraria um deploy, uma notificação ou uma publicação de artefato

Requisitos adicionais:
- Versões atualizadas de todas as actions (`@v4`, `@v5`)
- Todos os secrets passados corretamente com `env:`
- `black` e `autoflake` presentes no `requirements.txt`
- Comentários explicando a lógica de cada job

In [ ]:
name: CI — Bella Tavola

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  qualidade:
    runs-on: ubuntu-latest

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependências
        run: |
          python -m pip install --upgrade pip
          pip install -r aula_01_e02_fastapi/requirements.txt

      - name: Diagnóstico do ambiente
        run: |
          echo "=== Python ==="
          python --version
          echo "=== Dependências instaladas ==="
          pip list

      - name: Verificar formatação
        run: |
          cd aula_01_e02_fastapi
          black --check .

      - name: Verificar imports não utilizados
        run: |
          cd aula_01_e02_fastapi
          autoflake --check --remove-all-unused-imports -r .

      - name: Rodar testes smoke e validacao
        run: |
          cd aula_01_e02_fastapi
          pytest tests/ -m "smoke or validacao" -v --tb=short

  integracao:
    runs-on: ubuntu-latest
    needs: qualidade
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependências
        run: |
          python -m pip install --upgrade pip
          pip install -r aula_01_e02_fastapi/requirements.txt

      - name: Cache do modelo Hugging Face
        uses: actions/cache@v4
        with:
          path: ~/.cache/huggingface
          key: hf-model-v1-${{ hashFiles('aula_01_e02_fastapi/requirements.txt') }}

      - name: Verificar autenticação com Hugging Face
        run: python -c "import os; assert os.environ.get('HF_TOKEN'), 'HF_TOKEN não configurado'"
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}

      - name: Rodar testes de integração
        run: |
          cd aula_01_e02_fastapi
          pytest tests/test_modelo.py -m integracao -v --tb=short
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_REPO_ID: anaclaragr05/mlops-fraud-v1

💡 Gabarito

In [ ]:
# @title
# .github/workflows/ci.yml — versão final
#
# name: CI — Bella Tavola
#
# on:
#   push:
#     branches: [main]
#   pull_request:
#     branches: [main]
#
# jobs:
#   # ─────────────────────────────────────────────────────
#   # JOB 1: qualidade
#   # Roda em todo push e PR — feedback rápido para o desenvolvedor.
#   # Não depende de secrets nem de recursos externos.
#   # Requer: black e autoflake no requirements.txt
#   # ─────────────────────────────────────────────────────
#   qualidade:
#     runs-on: ubuntu-latest
#     steps:
#       - name: Baixar o código
#         uses: actions/checkout@v4
#       - name: Configurar Python
#         uses: actions/setup-python@v5
#         with:
#           python-version: "3.11"
#       - name: Instalar dependências
#         run: |
#           pip install --upgrade pip
#           pip install -r requirements.txt
#       - name: Diagnóstico do ambiente
#         run: |
#           python --version
#           pip list
#           test -f main.py          && echo "✅ main.py"          || echo "❌ main.py ausente"
#           test -f requirements.txt && echo "✅ requirements.txt" || echo "❌ requirements.txt ausente"
#           test -d tests/           && echo "✅ tests/"           || echo "❌ tests/ ausente"
#       - name: Verificar formatação
#         run: black --check .
#       - name: Verificar imports não utilizados
#         run: autoflake --check --remove-all-unused-imports -r .
#       - name: Rodar testes smoke
#         run: pytest tests/ -m smoke -v --tb=short
#
#   # ─────────────────────────────────────────────────────
#   # JOB 2: integracao
#   # Roda apenas no merge para main, depois de qualidade passar.
#   # Testa o modelo real baixado do Hugging Face Hub.
#   # Secrets não estão disponíveis em PRs de forks externos —
#   # por isso esse job é restrito a pushes diretos.
#   # ─────────────────────────────────────────────────────
#   integracao:
#     runs-on: ubuntu-latest
#     needs: qualidade
#     if: github.event_name == 'push' && github.ref == 'refs/heads/main'
#     steps:
#       - name: Baixar o código
#         uses: actions/checkout@v4
#       - name: Configurar Python
#         uses: actions/setup-python@v5
#         with:
#           python-version: "3.11"
#       - name: Instalar dependências
#         run: |
#           pip install --upgrade pip
#           pip install -r requirements.txt
#       - name: Cache do modelo Hugging Face
#         uses: actions/cache@v4
#         with:
#           path: ~/.cache/huggingface
#           key: hf-model-v1-${{ hashFiles('requirements.txt') }}
#       - name: Diagnóstico do ambiente
#         run: |
#           python --version
#           test -f model_utils.py && echo "✅ model_utils.py" || echo "❌ model_utils.py ausente"
#           if [ -n "$HF_TOKEN" ]; then echo "✅ HF_TOKEN definido"; else echo "❌ HF_TOKEN ausente"; fi
#         env:
#           HF_TOKEN: ${{ secrets.HF_TOKEN }}
#       - name: Verificar autenticação com Hugging Face
#         run: python -c "import os; assert os.environ.get('HF_TOKEN'), 'HF_TOKEN não configurado'"
#         env:
#           HF_TOKEN: ${{ secrets.HF_TOKEN }}
#       - name: Rodar testes de integração
#         run: pytest tests/ -m integracao -v --tb=short
#         env:
#           HF_TOKEN: ${{ secrets.HF_TOKEN }}
#
#   # ─────────────────────────────────────────────────────
#   # JOB 3: relatorio
#   # Roda após integração — gancho didático para demonstrar encadeamento.
#   # Em um projeto real, aqui entraria deploy, notificação ou
#   # publicação de artefato. Por ora, apenas exibe um resumo.
#   # ─────────────────────────────────────────────────────
#   relatorio:
#     runs-on: ubuntu-latest
#     needs: integracao
#     if: github.event_name == 'push' && github.ref == 'refs/heads/main'
#     steps:
#       - name: Baixar o código
#         uses: actions/checkout@v4
#       - name: Resumo do pipeline
#         run: |
#           echo "================================================"
#           echo "  Pipeline CI do Bella Tavola concluído ✅"
#           echo "================================================"
#           echo "  Branch:  ${{ github.ref_name }}"
#           echo "  Commit:  ${{ github.sha }}"
#           echo "  Autor:   ${{ github.actor }}"
#           echo "================================================"
#           echo "  Próximo passo: configurar deploy automático"
#           echo "================================================"

## Checklist do Bloco 6

Antes de encerrar o caderno, confirme que você consegue:

- [ ] Identificar qual step falhou em um log de pipeline
- [ ] Distinguir `ERROR` de import de `FAILED` de assert no pytest
- [ ] Localizar a causa raiz lendo o log — incluindo casos em que o erro visível é consequência de um step anterior
- [ ] Reconhecer os erros mais comuns pelo padrão da mensagem
- [ ] Separar erros objetivos de riscos de engenharia em um workflow
- [ ] Criar um ambiente virtual limpo para reproduzir falhas do CI localmente
- [ ] Remover o ambiente virtual nos sistemas operacionais que você usa
- [ ] Adicionar steps de diagnóstico informativo ao pipeline
- [ ] Montar um pipeline completo de três jobs com dependências e condições ✅

## Mapa do pipeline construído

Ao final deste caderno, o Bella Tavola tem o seguinte pipeline de CI:

```
Push ou PR para main
        ↓
  ┌─────────────┐
  │  qualidade  │  → formatação + imports + testes smoke
  └──────┬──────┘
         │ (só em push para main)
         ↓
  ┌─────────────┐
  │ integracao  │  → baixa modelo do Hub + testes de integração
  └──────┬──────┘
         │
         ↓
  ┌─────────────┐
  │  relatorio  │  → resumo + ponto de extensão para deploy
  └─────────────┘
```

## Checklist de competências — Semana 3 completa

Marque o que você consegue fazer ao final das três partes:

**Parte 1 — CI/CD e GitHub Actions**
- [ ] Explicar a diferença entre CI, CD (Entrega) e CD (Implantação)
- [ ] Ler e interpretar qualquer arquivo de workflow
- [ ] Escrever gatilhos, jobs e steps corretamente
- [ ] Identificar e corrigir erros de sintaxe YAML
- [ ] Fazer o primeiro pipeline ficar verde ✅

**Parte 2 — Testes automatizados com pytest**
- [ ] Criar testes com `TestClient` do FastAPI
- [ ] Usar fixtures e `conftest.py` para organizar os testes
- [ ] Escrever testes que verificam comportamentos relativos, não estado absoluto
- [ ] Usar parametrização e marcadores
- [ ] Configurar o pipeline para rodar subconjuntos de testes por evento

**Parte 3 — Integração com modelo e debugging**
- [ ] Configurar e usar secrets no GitHub Actions
- [ ] Baixar e testar o modelo do Hub no pipeline
- [ ] Usar cache e explicar quando ele é e não é invalidado
- [ ] Ler logs de pipeline e identificar causa raiz
- [ ] Distinguir erros objetivos de riscos de engenharia
- [ ] Reproduzir falhas do CI localmente com ambiente virtual limpo
- [ ] Montar um pipeline completo de três jobs com dependências e condições ✅